# M11.7 · Çok kollu haydutlar — pişmanlık, UCB, Thompson örneklemesi

Tek durumlu RL: geçiş yok, yalnız rastgele ödül fonksiyonu öğrenilir — keşif–sömürü ikileminin en saf hâli (klinik deneyler, reklam yerleştirme, web kişiselleştirme). Pişmanlık (regret) tanımı, sabit ε'un doğrusal ve $\varepsilon_t\propto1/t$'nin logaritmik pişmanlığı, iyimserlik ilkesi, Hoeffding eşitsizliğinden UCB1, Bayesçi haydutlar ve Beta–Bernoulli eşleniği ile Thompson örneklemesi (k örnekli sürüm), gradyan haydut ve Lai–Robbins alt sınırı. Saf Python'da beş algoritma, NumPy'da Sutton'ın 10 kollu test yatağı, PyTorch'ta Thompson'da örnek sayısı k'nın etkisi ve binlerce deneyin paralel simülasyonu.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-7-bandits.html](https://m-gonul.github.io/pytorch-lab/m11-7-bandits.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_7_bandits.py
CHECKS = {
 "b": "from labcheck import no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t():\n        probs = [0.9, 0.8, 0.7, 0.5, 0.3]\n        rows, res = [], {}\n        for name, pol, kw in [(\"açgözlü\", \"eps\", {\"eps\": 0.0}), (\"ε = 0.1\", \"eps\", {}), (\"ε_t = min(1, 5/t)\", \"eps_decay\", {}), (\"UCB1\", \"ucb\", {}), (\"Thompson\", \"ts\", {})]:\n            runs = [need(ns, \"run\")(pol, probs, 4000, s, **kw) for s in range(20)]\n            ensure(all(len(r) == 4000 for r in runs), \"uzunluk\")\n            r1 = sum(r[999] for r in runs) / 20\n            r2 = sum(r[1999] for r in runs) / 20\n            r4 = sum(r[-1] for r in runs) / 20\n            res[name] = (r1, r4, max(r[-1] for r in runs), (r4 - r2) / max(r2 - r1, 1e-9))\n            rows.append([name, f\"{r1:.1f}\", f\"{r4:.1f}\", f\"{res[name][3]:.2f}\", f\"{res[name][2]:.0f}\"])\n        ensure(res[\"Thompson\"][1] < min(v[1] for k, v in res.items() if k != \"Thompson\"), \"Thompson en düşük pişmanlık\")\n        ensure(res[\"ε = 0.1\"][3] > 1.7 and res[\"UCB1\"][3] < 1.4, \"sabit ε doğrusal, UCB1 logaritmik büyümeli\")\n        ensure(res[\"açgözlü\"][2] > 1000, \"açgözlü bazı deneylerde kötü kola takılmalı\")\n        R_.table(\"Kollar 0.9, 0.8, 0.7, 0.5, 0.3 — 20 deney ortalama pişmanlık\", [\"yöntem\", \"t = 1000\", \"t = 4000\", \"artış oranı\", \"en kötü deney\"], rows,\n                 note=\"Artış oranı = [L(4000) − L(2000)] / [L(2000) − L(1000)]: doğrusal pişmanlıkta ≈ 2, logaritmikte ≈ ln 2 / ln 2 = 1. UCB1'in √(2 ln t/n) sınırı muhafazakârdır: başta çok keşfeder, uzun vadede ε'u geçer.\")\n    R_.test(\"beş algoritma\", t)\n",
 "c": "import numpy as np\nfrom labcheck import max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız adım döngüsü\", lambda: (max_loops(src, 1, func=\"testbed\"), max_loops(src, 1, func=\"gradient_bandit\")))\n\n    def tb():\n        rows, res = [], {}\n        for name, kw in [(\"ε = 0 (açgözlü)\", dict(eps=0.0)), (\"ε = 0.01\", dict(eps=0.01)), (\"ε = 0.1\", dict(eps=0.1)),\n                         (\"iyimser Q0 = 5, ε = 0, α = 0.1\", dict(eps=0.0, Q0=5.0, alpha=0.1)), (\"gerçekçi Q0 = 0, ε = 0.1, α = 0.1\", dict(eps=0.1, alpha=0.1))]:\n            r, o = need(ns, \"testbed\")(2000, 1000, seed=1, **kw)\n            res[name] = (r[-100:].mean(), o[-100:].mean(), o[:50].mean())\n            rows.append([name, f\"{res[name][0]:.2f}\", f\"{100 * res[name][1]:.0f}%\", f\"{100 * res[name][2]:.0f}%\"])\n        ensure(res[\"ε = 0.1\"][0] > res[\"ε = 0.01\"][0] > res[\"ε = 0 (açgözlü)\"][0], \"Sutton Şekil 2.2 sıralaması (1000 adımda)\")\n        ensure(res[\"iyimser Q0 = 5, ε = 0, α = 0.1\"][1] > res[\"gerçekçi Q0 = 0, ε = 0.1, α = 0.1\"][1] and res[\"iyimser Q0 = 5, ε = 0, α = 0.1\"][2] < res[\"gerçekçi Q0 = 0, ε = 0.1, α = 0.1\"][2], \"iyimser başlangıç: başta kötü, sonra iyi\")\n        for name, kw in [(\"gradyan α = 0.1, taban çizgili\", dict(alpha=0.1)), (\"gradyan α = 0.1, taban çizgisiz\", dict(alpha=0.1, baseline=False))]:\n            r, o = need(ns, \"gradient_bandit\")(2000, 1000, seed=1, **kw)\n            res[name] = (r[-100:].mean(), o[-100:].mean(), o[:50].mean())\n            rows.append([name + \" (q* ortalaması 4)\", f\"{res[name][0]:.2f}\", f\"{100 * res[name][1]:.0f}%\", f\"{100 * res[name][2]:.0f}%\"])\n        ensure(res[\"gradyan α = 0.1, taban çizgili\"][1] > res[\"gradyan α = 0.1, taban çizgisiz\"][1] + 0.15, \"taban çizgisi önemli (Şekil 2.5)\")\n        R_.table(\"10 kollu test yatağı, 2000 deney × 1000 adım\", [\"yöntem\", \"son 100 adım ort. ödül\", \"son 100 adım optimal\", \"ilk 50 adım optimal\"], rows,\n                 note=\"Taban çizgisi olmadan bütün ödüller ≈ 4 pozitif olduğu için seçilen her eylemin tercihi artar; varyans büyür ve öğrenme yavaşlar.\")\n    R_.test(\"test yatağı ve gradyan haydut\", tb)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def np_():\n        mu, var = need(ns, \"normal_posterior\")(torch.tensor([3.0, 0.0]), torch.tensor([3.0, 0.0]), 0.0, 1.0, 1.0)\n        close(mu.numpy(), torch.tensor([0.75, 0.0]).numpy(), what=\"sonsal ortalama\")\n        close(var.numpy(), torch.tensor([0.25, 1.0]).numpy(), what=\"sonsal varyans\")\n        g = torch.Generator().manual_seed(0)\n        x = 2.0 + 0.5 * torch.randn(10000, generator=g, dtype=torch.float64)\n        mu, var = ns[\"normal_posterior\"](x.sum(), 10000, 0.0, 100.0, 0.25)\n        ensure(abs(float(mu) - 2.0) < 0.02 and float(var) < 1e-4, \"çok veride MLE'ye yakınsar\")\n    R_.test(\"normal_posterior\", np_)\n\n    def k():\n        probs = [0.9, 0.8, 0.7, 0.5, 0.3]\n        rows, res = [], {}\n        for kk in (1, 10, 100):\n            r = need(ns, \"thompson\")(probs, 1000, 200, kk, 3)\n            ensure(r.shape == (1000,) and bool((r[1:] >= r[:-1] - 1e-12).all()), \"artan kümülatif pişmanlık\")\n            res[kk] = r\n            rows.append([kk, f\"{float(r[99]):.1f}\", f\"{float(r[-1]):.1f}\"])\n        ensure(float(res[1][-1]) < float(res[100][-1]), \"k = 1 daha iyi keşfetmeli\")\n        rg = need(ns, \"gaussian_thompson\")([1.0, 0.5, 0.0, -0.5], 1000, 200, 1.0, 3)\n        rows.append([\"Normal–Normal (k = 1, kollar 1, 0.5, 0, −0.5; σ = 1)\", f\"{float(rg[99]):.1f}\", f\"{float(rg[-1]):.1f}\"])\n        ensure(float(rg[-1]) < 100, \"Gauss Thompson öğrenmeli\")\n        R_.table(\"Thompson örneklemesi, 200 deney\", [\"k\", \"t = 100\", \"t = 1000\"], rows,\n                 note=\"k büyüdükçe R̂ sonsal ortalamaya yaklaşır: rastgelelik — ve keşif — kaybolur; bazı deneyler alt-optimal kola takılır ve pişmanlık uzun vadede artar.\")\n    R_.test(\"k'nın etkisi ve Gauss Thompson\", k)\n",
 "e": "import math\nimport random\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef loop_ucb(probs, T, n_runs):\n    K = len(probs)\n    for k in range(n_runs):\n        rng = random.Random(k)\n        n, s = [0] * K, [0.0] * K\n        for t in range(1, T + 1):\n            a = t - 1 if t <= K else max(range(K), key=lambda i: s[i] / n[i] + math.sqrt(2 * math.log(t) / n[i]))\n            r = 1 if rng.random() < probs[a] else 0\n            n[a] += 1\n            s[a] += r\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"vec_bandit\"))\n    probs = [0.9, 0.8, 0.7, 0.5, 0.3]\n\n    def lr():\n        close(need(ns, \"lai_robbins\")([0.5, 0.4]), 0.1 / (0.4 * math.log(0.8) + 0.6 * math.log(0.6 / 0.5)), what=\"Lai–Robbins sabiti\")\n    R_.test(\"lai_robbins\", lr)\n\n    def v():\n        rows, res = [], {}\n        for m in (\"eps\", \"ucb\", \"ts\"):\n            r = need(ns, \"vec_bandit\")(probs, 3000, 2000, m, 0.1, 1)\n            res[m] = r\n            rows.append([m, f\"{float(r[299]):.1f}\", f\"{float(r[-1]):.1f}\", f\"{float(r[-1]) / math.log(3000):.1f}\"])\n        lb = ns[\"lai_robbins\"](probs)\n        rows.append([\"Lai–Robbins alt sınır sabiti\", \"\", f\"{lb * math.log(3000):.1f}\", f\"{lb:.1f}\"])\n        ensure(float(res[\"ts\"][-1]) < float(res[\"ucb\"][-1]) and float(res[\"ts\"][-1]) < float(res[\"eps\"][-1]), \"Thompson en iyi\")\n        ensure(abs(float(res[\"eps\"][-1]) - float(res[\"eps\"][1999]) * 1.5) < 0.2 * float(res[\"eps\"][-1]), \"ε: doğrusal\")\n        t_vec = bench(lambda: ns[\"vec_bandit\"](probs, 3000, 2000, \"ucb\"), repeat=1, number=1)\n        t_loop = bench(lambda: loop_ucb(probs, 3000, 20), repeat=1, number=1) * 100\n        rows.append([\"süre: 2000 deney paralel (UCB)\", fmt_time(t_vec), \"\", \"\"])\n        rows.append([\"süre: deney döngüsü (tahmini)\", fmt_time(t_loop), \"\", \"\"])\n        R_.table(\"2000 deney × 3000 adım\", [\"yöntem\", \"pişmanlık t = 300\", \"t = 3000\", \"/ ln t\"], rows,\n                 note=\"Asimptotik alt sınır yalnız büyük t'de geçerlidir; Thompson'ın eğimi ona en yakın olandır (Chapelle & Li 2011).\")\n    R_.test(\"vec_bandit\", v)\n",
 "x1": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        for p, n, c in [(0.5, 100, 0.1), (0.5, 100, 0.2), (0.9, 100, 0.1), (0.5, 1000, 0.05), (0.99, 50, 0.05)]:\n            emp, bd = need(ns, \"coverage\")(p, n, c, 200000, 1)\n            ensure(emp <= bd + 0.005, f\"Hoeffding ihlal edilmemeli ({p}, {n}, {c})\")\n            rows.append([p, n, c, f\"{emp:.4f}\", f\"{bd:.4f}\"])\n        R_.table(\"P(|x̄ − p| ≥ c): deney vs Hoeffding\", [\"p\", \"n\", \"c\", \"deneysel\", \"2e^{−2nc²}\"], rows,\n                 note=\"Hoeffding dağılımdan bağımsızdır, bu yüzden özellikle p uçlarda (varyans küçük) gevşektir; UCB-V, KL-UCB gibi varyant'lar bu boşluğu kapatır.\")\n    R_.test(\"Hoeffding\", t)\n",
 "x2": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        a = need(ns, \"nonstationary\")(1000, 5000, None, 1)\n        b = ns[\"nonstationary\"](1000, 5000, 0.1, 1)\n        ensure(b[-500:].mean() > a[-500:].mean() + 0.1, \"sabit α durağan olmayan durumda daha iyi\")\n        R_.table(\"Durağan olmayan haydut, optimal eylem oranı (1000 deney)\", [\"adım\", \"örnek ortalaması\", \"α = 0.1\"],\n                 [[k, f\"{100 * a[k - 100:k].mean():.0f}%\", f\"{100 * b[k - 100:k].mean():.0f}%\"] for k in (500, 1000, 2500, 5000)],\n                 note=\"1/n adımları eski ödülleri eşit ağırlıkla tutar; sabit α üstel ağırlıklı yakın-geçmiş ortalamasıdır ve değişen q*'ı izler — ama Robbins–Monro'nun Σα² < ∞ koşulunu bilerek bozar.\")\n    R_.test(\"durağan olmayan haydut\", t)\n",
 "x3": "import random\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = random.Random(5)\n        ctr = [0.01 + 0.04 * rng.random() for _ in range(20)]\n        T = 20000\n        best = max(ctr) * T\n        u = sum(need(ns, \"ads\")(\"ucb\", ctr, T, s) for s in range(3)) / 3\n        th = sum(ns[\"ads\"](\"ts\", ctr, T, s) for s in range(3)) / 3\n        rnd = sum(ctr) / 20 * T\n        ensure(th > u, \"küçük tıklama oranlarında Thompson UCB1'den iyi\")\n        p, loss = need(ns, \"ab_test\")(30, 1000, 45, 1000)\n        ensure(0.93 < p < 0.98 and loss < 0.001, \"A/B: P(B > A) ≈ 0.96\")\n        p2, _ = ns[\"ab_test\"](30, 1000, 30, 1000)\n        ensure(abs(p2 - 0.5) < 0.01, \"eşit veride ≈ 0.5\")\n        R_.table(\"20 reklam, tıklama oranı %1–%5, 20 000 gösterim (3 tohum)\", [\"strateji\", \"tıklama\"],\n                 [[\"rastgele (beklenen)\", f\"{rnd:.0f}\"], [\"UCB1\", f\"{u:.0f}\"], [\"Thompson\", f\"{th:.0f}\"], [\"en iyi reklam hep (oracle)\", f\"{best:.0f}\"]],\n                 note=f\"UCB1'in Hoeffding terimi [0,1] aralığını varsaydığı için %3'lük oranlarda aşırı keşfeder. A/B testi (30/1000 vs 45/1000): P(B > A) = {p:.3f}, B'yi seçmenin beklenen kaybı {loss:.5f}.\")\n    R_.test(\"reklam ve A/B\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_7_bandits as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Haydut:** tek durum $S=\{s\}$, eylemler (kollar) $A$, ödüller (genelde [0,1]'e ölçekli). Geçiş yok; yalnız rastgele ödül fonksiyonu öğrenilir. Ad: kumar makinesi ("tek kollu haydut"). Uygulamalar: klinik deney tasarımı, çevrimiçi reklam (tıklama oranı), web kişiselleştirme, oyunlar, ağ yönlendirme. Basit tanımlı ama genel durumda bilinen izlenebilir optimal çözüm yok.

**Pişmanlık:** $r^*=\max_aR(a)$; $\mathrm{loss}(a)=r^*-R(a)$; $\mathrm{Loss}_n=\sum_{t=1}^n\mathbb E[r^*-R(a_t)]$.

- Açgözlü: takılabilir. ε-açgözlü sabit ε: $P(a_t\ne a^*)\approx\varepsilon$ → $\mathrm{Loss}_n=O(n)$ (doğrusal). $\varepsilon_t\propto1/t$: $\sum1/t\approx\ln n+0.5772$ → $O(\log n)$.
- **İyimserlik:** her kol için gerçeği yüksek olasılıkla aşan bir üst sınır $\mathrm{UB}_n(a)$ al ve en büyüğünü seç. Sınır gerçeğe yakınsıyorsa, alt-optimal bir kola yakınsamak çelişki doğurur. Hoeffding: $P(R(a)\le\tilde R(a)+c)\ge1-e^{-2n_ac^2}$; $\delta_n=1/n^4$ ile $c=\sqrt{2\ln n/n_a}$ → **UCB1**, $O(\log n)$ pişmanlık.
- **Bayesçi:** $\theta$ üzerinde önsel, gözlemlerle sonsal $P(\theta|r_{1:n})\propto P(\theta)\prod P(r_i|\theta)$. Eşlenik çiftlerde (Beta–Bernoulli, Normal–Normal) sonsal kapalı form. **Thompson:** her koldan sonsaldan k örnek çek, ortala, en büyüğünü oyna (pratikte k = 1). Rastgelelik keşfi sağlar; $O(\log n)$ ve pratikte genelde en iyisi.
- **Alt sınır** (Lai–Robbins): her tutarlı algoritma için $\liminf\mathrm{Loss}_n/\ln n\ge\sum_{a:\Delta_a>0}\Delta_a/\mathrm{KL}(R(a)\,\|\,r^*)$.

## 1 · Elle

<!-- çözüm -->

UCB1: $\sqrt{2\ln100/10}=\sqrt{0.92103}=0.95971$ → 1.5597 (kol ortalaması 0.6 olsa da "iyimser" değer 1'in üstünde: az denenmiş).

Beta: $\mathrm{Beta}(1,1)$ düzgün; her yazı α'yı, her tura β'yı 1 artırır → Beta(4, 2), ortalama $\alpha/(\alpha+\beta)=2/3$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Pişmanlık: kollar 0.9, 0.8, 0.5; sırasıyla 50, 30, 20 kez çekildi**

- `reg`: Beklenen kümülatif pişmanlık
- `eps`: Sabit ε = 0.1, büyük t'de adım başı pişmanlık (rastgele seçim 3 koldan)
- `har`: $\sum_{t=1}^{100}1/t$ (≈ ln 100 + 0.5772)

**Hoeffding ve UCB1**

- `hoe`: $n=100$, $c=0.1$: $2e^{-2nc^2}$
- `ucb`: UCB1: ortalama 0.6, $n=100$, $n_a=10$

**Beta–Bernoulli: önsel Beta(1,1), 3 tura 1 yazı (tura = ödül 1)**

- `ba`: Sonsal α
- `bm`: Sonsal ortalama

In [ ]:
cevaplar = dict(
    reg=None,
    eps=None,
    har=None,
    hoe=None,
    ucb=None,
    ba=None,
    bm=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"reg":[11,0],"eps":[0.016666666666666666,1e-9],"har":[5.187377517639621,0.000001],"hoe":[0.2706705664732254,0.000001],"ucb":[1.5597051061,0.000001],"ba":[4,0],"bm":[0.6666666666666666,1e-9]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, Bernoulli kolları `probs`.

- `run(policy, probs, T, seed=0, eps=0.1, c=5.0)` → kümülatif **beklenen** pişmanlık listesi ($\sum_t r^*-p_{a_t}$). `policy`:
  - `"eps"` — sabit ε (ε = 0 açgözlü); eşitlik/ilk adım için rastgele; tahmin 0 ile başlar.
  - `"eps_decay"` — $\varepsilon_t=\min(1,c/t)$.
  - `"ucb"` — ilk K adımda her kolu bir kez çek, sonra $\arg\max\,\bar r_a+\sqrt{2\ln t/n_a}$.
  - `"ts"` — Beta(1 + yazı, 1 + tura)'dan `rng.betavariate` ile birer örnek.

In [ ]:
%%writefile stage_b.py
import math
import random


def run(policy, probs, T, seed=0, eps=0.1, c=5.0):
    ...


print(run("ts", [0.9, 0.8, 0.5], 1000)[-1])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

**10 kollu test yatağı** (Sutton Bölüm 2), NumPy, tüm deneyler aynı anda; yalnız adım döngüsü.

Her deneyde $q_*(a)\sim N(\text{offset},1)$, ödül $N(q_*(a),1)$.

- `testbed(n_runs, n_steps, eps, Q0=0.0, alpha=None, seed=0)` → `(ortalama ödül (n_steps,), optimal eylem oranı (n_steps,))`; `alpha=None` örnek ortalaması, yoksa sabit α; açgözlü eşitlikler rastgele.
- `gradient_bandit(n_runs, n_steps, alpha, baseline=True, seed=0, offset=4.0)` — tercihler $H$, $\pi=\mathrm{softmax}(H)$; $H\leftarrow H+\alpha(r-\bar r)(\mathbb 1_a-\pi)$, $\bar r$ ortalama ödül (taban çizgisi yoksa 0). Aynı çıktılar.

In [ ]:
%%writefile stage_c.py
import numpy as np


def testbed(n_runs, n_steps, eps, Q0=0.0, alpha=None, seed=0):
    ...


def gradient_bandit(n_runs, n_steps, alpha, baseline=True, seed=0, offset=4.0):
    ...


r, opt = testbed(200, 500, 0.1)
print(r[-50:].mean(), opt[-50:].mean())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch, Bayesçi haydutlar.

- `normal_posterior(sum_r, n, mu0, var0, noise_var)` — Normal önsel, Normal olabilirlik (bilinen gürültü): $\text{kesinlik}=1/\sigma_0^2+n/\sigma^2$, ortalama $=(\mu_0/\sigma_0^2+\sum r/\sigma^2)/\text{kesinlik}$ → `(mean, var)` (tensörler, şekil korunur).
- `thompson(probs, T, n_runs, k=1, seed=0)` — Beta–Bernoulli, her kol için **k** sonsal örneğin ortalaması; deneyler paralel; ortalama kümülatif pişmanlık `(T,)`.
- `gaussian_thompson(means, T, n_runs, noise_sd=1.0, seed=0)` — Normal–Normal (önsel N(0, 1)); pişmanlık `(T,)`.

In [ ]:
%%writefile stage_d.py
import torch


def normal_posterior(sum_r, n, mu0, var0, noise_var):
    ...


def thompson(probs, T, n_runs, k=1, seed=0):
    ...


def gaussian_thompson(means, T, n_runs, noise_sd=1.0, seed=0):
    ...


print(thompson([0.9, 0.8, 0.5], 200, 100)[-1])

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Vektörel simülasyon, `n_runs` deney aynı anda (PyTorch).

- `vec_bandit(probs, T, n_runs, method, eps=0.1, seed=0)` → `(T,)` ortalama kümülatif pişmanlık; `method ∈ {"eps", "ucb", "ts"}` (b ile aynı kurallar).
- `lai_robbins(probs)` → $\sum_{a:\Delta_a>0}\Delta_a/\mathrm{KL}(p_a\|p^*)$ (Bernoulli KL).

In [ ]:
%%writefile stage_e.py
import math
import torch


def vec_bandit(probs, T, n_runs, method, eps=0.1, seed=0):
    ...


def lai_robbins(probs):
    ...


print(vec_bandit([0.9, 0.8, 0.5], 500, 1000, "ucb")[-1])

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Hoeffding sınırı ne kadar sıkı?

- `coverage(p, n, c, n_trials, seed=0)` — `n_trials` kez n Bernoulli(p) örneğin ortalamasını al (NumPy, döngüsüz: `binomial`), $P(|\bar x-p|\ge c)$'nin deneysel değeri ve Hoeffding sınırı $2e^{-2nc^2}$ → `(emp, bound)`.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def coverage(p, n, c, n_trials, seed=0):
    ...


print(coverage(0.5, 100, 0.1, 100000))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Durağan olmayan haydut: örnek ortalaması vs sabit α

**Durağan olmayan** 10 kollu haydut (Sutton Alıştırma 2.5): tüm $q_*$ 0'dan başlar ve her adımda $N(0,0.01^2)$ rastgele yürür. ε = 0.1.

- `nonstationary(n_runs, n_steps, alpha=None, seed=0)` → optimal eylem oranı `(n_steps,)` (NumPy, deneyler paralel); `alpha=None` örnek ortalaması.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def nonstationary(n_runs, n_steps, alpha=None, seed=0):
    ...


print(nonstationary(200, 1000).mean())

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Reklam yerleştirme ve Bayesçi A/B testi

**Reklam:** 20 reklam, tıklama oranları %1–%5. Her gösterim bir çekiş.

- `ads(policy, ctr, T, seed=0)` — `"ucb"` ya da `"ts"`; toplam tıklama.
- `ab_test(clicks_a, n_a, clicks_b, n_b, n_samples=200000, seed=0)` — Beta(1,1) önselle $P(p_B>p_A)$ (Monte Carlo) ve B'yi seçmenin beklenen kaybı $\mathbb E[\max(p_A-p_B,0)]$.

In [ ]:
%%writefile extra_x3.py
import math
import random
import numpy as np


def ads(policy, ctr, T, seed=0):
    ...


def ab_test(clicks_a, n_a, clicks_b, n_b, n_samples=200000, seed=0):
    ...


print(ab_test(30, 1000, 45, 1000))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Auer ve ark. (2002): UCB1'in sonlu zaman analizi

[Auer, Cesa-Bianchi, Fischer — Finite-time Analysis of the Multiarmed Bandit Problem](https://doi.org/10.1023/A:1013689704352)

UCB1, UCB1-Tuned, UCB2 ve ε_n-greedy'yi (ε_n = min(1, cK/(d²n))) makalenin Bernoulli deney düzenekleriyle (Tablo 1: 1–14 numaralı dağılımlar) karşılaştır; Teorem 1'deki pişmanlık üst sınırını $8\sum_{i:\Delta_i>0}\ln n/\Delta_i+(1+\pi^2/3)\sum\Delta_i$ deneysel pişmanlıkla aynı grafikte çiz.


**Adımlar**

1. e aşamasındaki vektörel simülasyonu genişlet (100 deney × 100 000 adım).
2. Log-x ekseninde pişmanlık.

**Kabul ölçütleri**

- UCB1 sınırın altında kalır; sınır gevşektir (≈ 5–10 kat).
- UCB1-Tuned çoğu düzenekte en iyisidir (makale Şekil 2–5 nitel).

## Makale ödevi: Chapelle & Li (2011): Thompson örneklemesinin deneysel değerlendirmesi

[Chapelle & Li — An Empirical Evaluation of Thompson Sampling (NeurIPS 2011)](https://papers.nips.cc/paper/2011/hash/e53a0a2978c28872a4505bdb51db06dc-Abstract.html)

Makalenin Bölüm 3 simülasyonunu yeniden üret: K ∈ {10, 100}, en iyi kol 0.5, diğerleri 0.5 − ε (ε ∈ {0.02, 0.1}), 10⁷ adıma kadar; Thompson, UCB ve Lai–Robbins asimptotik alt sınırını karşılaştır. Gecikmeli geri bildirim deneyini (Tablo 1) ve sonsal yapay olarak "sıkıştırma"nın (α ile ölçekleme) etkisini de dene.


**Adımlar**

1. Beta örneklemesini vektörel yap; 10⁷ adım için yığın halinde ilerle.

**Kabul ölçütleri**

- Thompson eğrisinin eğimi Lai–Robbins sınırına yaklaşır, UCB'den düşük kalır.
- Gecikmede Thompson UCB'den daha az etkilenir.

## Kaynaklar

- [Auer, Cesa-Bianchi, Fischer — Finite-time Analysis of the Multiarmed Bandit Problem (Machine Learning 2002)](https://doi.org/10.1023/A:1013689704352) — ©
- [Chapelle & Li — An Empirical Evaluation of Thompson Sampling (NeurIPS 2011)](https://papers.nips.cc/paper/2011/hash/e53a0a2978c28872a4505bdb51db06dc-Abstract.html) — ©
- [Lai & Robbins — Asymptotically Efficient Adaptive Allocation Rules (Adv. Appl. Math. 1985)](https://doi.org/10.1016/0196-8858(85)90002-8) — ©
- [Lattimore & Szepesvári — Bandit Algorithms (Cambridge 2020)](https://tor-lattimore.com/downloads/book/book.pdf) — ücretsiz PDF
- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı), Bölüm 2](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0